# Démo 3 : feature extraction et fine-tuning d'un ResNet

Objectif : charger un ResNet50 pré-entraîné sur ImageNet, remplacer sa dernière couche par une tête à 10 classes, entraîner la tête seule (feature extraction), puis dégeler le dernier bloc avec un learning rate plus faible (fine-tuning partiel).

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- charger un ResNet50 pré-entraîné sur ImageNet et ses transforms ;
- la feature extraction : geler le réseau et n'entraîner qu'une nouvelle tête ;
- le fine-tuning : dégeler le dernier bloc, avec un learning rate plus faible.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [1]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge le jeu d'entraînement, puis celui de
  test, s'ils ne sont pas déjà là ;
- `%%capture` : masque la barre de progression du téléchargement.

In [2]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Les transforms du modèle

**Chaque jeu de poids pré-entraînés fournit sa transform.**

Le ResNet a été entraîné sur des photos ImageNet de 224 x 224 pixels, en couleur, normalisées d'une
certaine façon. Pour réutiliser ses poids, il faut lui présenter les images exactement de la même façon.

Le code :
- `ResNet50_Weights.DEFAULT` : les meilleurs poids disponibles pour ResNet50 ;
- `weights.transforms()` : redimensionnement, recadrage et normalisation d'origine ;
- `v2.Grayscale(num_output_channels=3)` : le réseau attend 3 canaux ; on recopie le canal gris trois fois.

In [3]:
import torch
from torchvision.models import ResNet50_Weights, resnet50
from torchvision.transforms import v2

weights = ResNet50_Weights.DEFAULT
preprocess = v2.Compose([v2.Grayscale(num_output_channels=3), weights.transforms()])
print(preprocess)

In [5]:
print(resnet50(weights=weights))

La transform duplique le canal gris en trois canaux, redimensionne à 232, recadre au centre en 224 x 224 et normalise avec la moyenne et l'écart-type d'ImageNet, canal par canal.

## 3. Des données réduites

**2000 images d'entraînement et 500 de validation, pour que la démo reste courte.**

Un ResNet50 sur des images de 224 x 224 demande beaucoup plus de calcul que nos petits CNN.

Le code :
- `Subset(..., range(2000))` : les 2000 premières images ;
- `DataLoader` : des lots de 32 pour l'entraînement, de 64 pour la validation.

In [4]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

train_ds = Subset(datasets.FashionMNIST("data", train=True, transform=preprocess), range(2000))
val_ds = Subset(datasets.FashionMNIST("data", train=False, transform=preprocess), range(500))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
print(len(train_ds), len(val_ds), train_ds[0][0].shape)

2000 images d'entraînement, 500 de validation. Chaque petite image grise devient un tenseur `torch.Size([3, 224, 224])`.

## 4. Le modèle pré-entraîné

**Le ResNet50 et ses poids appris sur ImageNet.**

ResNet50 est un CNN de 50 couches, entraîné sur plus d'un million de photos réparties en 1000
classes. Ses couches ont appris à reconnaître bords, textures, formes : des features utiles bien au-delà
d'ImageNet.

Le code :
- `resnet50(weights=weights)` : crée le réseau et charge les poids, téléchargés la première fois ;
- `model.fc` : la dernière couche, qui produit les scores des 1000 classes d'ImageNet.

In [6]:
model = resnet50(weights=weights, progress=False)
print(model.fc)
print(sum(p.numel() for p in model.parameters()), "paramètres")

La dernière couche est `Linear(in_features=2048, out_features=1000)` : 1000 classes d'ImageNet. Le modèle compte 25557032 paramètres, bien trop pour les entraîner avec quelques milliers d'images.

**À retenir** : on ne réentraîne pas des millions de paramètres avec quelques milliers d'images ; on réutilise.

## 5. Geler le backbone, remplacer la tête

**On gèle tout le réseau, puis on remplace sa dernière couche par une tête neuve à 10 sorties.**

Le **backbone** (tout le réseau sauf la tête) garde ce qu'il sait ; seule la tête apprend nos 10
classes. C'est la **feature extraction**.

Le code :
- `p.requires_grad_(False)` pour chaque paramètre : ils ne recevront plus de gradient ;
- `nn.Linear(model.fc.in_features, 10)` : une tête neuve, dont les paramètres sont entraînables par défaut ;
- le décompte des paramètres entraînables vérifie que seule la tête apprendra.

In [8]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for p in model.parameters():
    p.requires_grad_(False)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)
print(sum(p.numel() for p in model.parameters() if p.requires_grad), "paramètres entraînables")

Seuls les 20490 paramètres de la nouvelle tête sont entraînables : les poids et les biais de la tête à 10 sorties.

**Piège fréquent** : geler le réseau mais donner tous les paramètres à l'optimiseur ; c'est sans effet ici, mais cela cache l'intention. On lui donne seulement la tête.

## 6. Mesurer la loss et l'accuracy

**La fonction `evaluate` du jour 1 : la loss moyenne et la part d'images bien classées.**

In [9]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Rien à afficher : la fonction est définie.

## 7. Entraîner quelques époques (fourni)

**La boucle habituelle, avec l'optimiseur passé en argument.**

C'est l'optimiseur qui décide quels paramètres bougent ; on pourra ainsi réutiliser la même
fonction pour les deux phases. Après chaque époque, une ligne donne la loss et l'accuracy sur
l'entraînement (`loss`, `acc`) puis sur la validation (`val_loss`, `val_acc`), et les mêmes mesures sont
écrites pour TensorBoard sous `runs/transfer/<nom>`.

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def entrainer(model, optimizer, n_epochs, nom):
    loss_fn = nn.CrossEntropyLoss()
    writer = SummaryWriter(f"runs/transfer/{nom}")
    for epoch in range(1, n_epochs + 1):
        model.train()
        for images, labels in train_loader:
            loss = loss_fn(model(images.to(device)), labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_loader, loss_fn)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        print(f"époque {epoch} : train_loss {train_loss:.3f}  train_acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
    writer.close()

Rien à afficher : la fonction est définie ; on l'appelle pour les deux phases.

## 8. Feature extraction : la tête seule

**Phase 1 : on n'entraîne que la tête.**

Le code :
- `torch.optim.Adam(model.fc.parameters(), lr=1e-3)` : l'optimiseur ne reçoit que les paramètres de la tête ;
- deux époques sur les 2000 images.

In [11]:
torch.manual_seed(0)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
N_EPOCHS = 2
nom = "tete-seule"
entrainer(model, optimizer, n_epochs=N_EPOCHS, nom=nom)

Avec la tête seule, l'accuracy de validation atteint 0.582 puis 0.736 en deux époques : les features d'ImageNet, apprises sur des photos en couleur, décrivent déjà en partie ces vêtements en niveaux de gris.

**À retenir** : la feature extraction donne vite un résultat correct, avec très peu de paramètres à entraîner.

## 9. Dégeler le dernier bloc

**Phase 2 : on dégèle `layer4`, le dernier bloc du ResNet.**

Les derniers blocs d'un réseau pré-entraîné contiennent les features les plus spécifiques à sa tâche
d'origine ; les adapter à nos images peut améliorer le résultat. C'est le **fine-tuning** partiel.

Le code :
- `p.requires_grad_(True)` pour les paramètres de `model.layer4` ;
- le décompte des paramètres entraînables.

In [12]:
for p in model.layer4.parameters():
    p.requires_grad_(True)
print(sum(p.numel() for p in model.parameters() if p.requires_grad), "paramètres entraînables")

14985226 paramètres deviennent entraînables : ceux de `layer4` et de la tête.

## 10. Fine-tuning avec un learning rate différentiel

**L'optimiseur reçoit deux groupes de paramètres, chacun avec son learning rate.**

`layer4` a déjà de bons poids ; de trop grands pas les détruiraient. On lui donne un learning rate
dix fois plus faible que celui de la tête, qui part de zéro.

Le code :
- une liste de deux dictionnaires : `{"params": ..., "lr": ...}` pour `layer4`, puis pour la tête ;
- deux nouvelles époques.

In [13]:
optimizer = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])
N_EPOCHS = 2
nom = "fine-tuning"
torch.manual_seed(0)
entrainer(model, optimizer, n_epochs=N_EPOCHS, nom=nom)

Avec un learning rate dix fois plus faible pour `layer4`, l'accuracy de validation monte à 0.818 puis 0.826 : adapter le dernier bloc apporte un gain net par rapport à la tête seule.

**Essayez** : donnez à `layer4` le même learning rate que la tête (1e-3) et comparez.

## 11. Les deux phases dans TensorBoard

**TensorBoard affiche les courbes écrites pendant l'entraînement, comme dans la démo 5.**

Le code :
- `%load_ext tensorboard` : charge l'extension dans le notebook ;
- `%tensorboard --logdir runs/transfer` : affiche les courbes de ce dossier ; chaque run a sa couleur, et on peut
  les cocher ou les décocher pour les comparer.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/transfer

Dans Colab, le tableau de bord de TensorBoard s'affiche sous la cellule ; ce notebook enregistré n'en garde pas l'image. Les deux phases y apparaissent comme deux runs, la tête seule puis le fine-tuning.

## À retenir de cette démo

- Un réseau pré-entraîné apporte des features génériques : on remplace seulement sa dernière couche par une tête adaptée à nos classes.
- Geler (`requires_grad_(False)`) empêche des paramètres d'apprendre ; l'optimiseur ne reçoit que ceux qu'on veut entraîner.
- Pour affiner des couches pré-entraînées, un learning rate plus faible : des groupes de paramètres dans l'optimiseur.